In [ ]:
from dotenv import load_dotenv
from pathlib import Path

c:\Users\paris\Documents\Cours Centrale 3A\technical-test\injury-prediction\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Database handling
## Nutrition data

To test the food calorie estimation model :

In [ ]:
from utils.calorie_estimation import CalorieEstimation

# Initialize CalorieEstimation using the utility class
model = CalorieEstimation()

# Predict calories for a food image
photo_path = r"data\p01\food-images\IMG_8916.jpeg"
calories = model.predict(photo_path)
print(f"Estimated: {calories:.0f} calories")

[INFO] Downloading/Verifying CalorieCLIP model files locally...


Fetching 19 files: 100%|██████████| 19/19 [00:00<00:00, 142.33it/s]
c:\Users\paris\Documents\Cours Centrale 3A\technical-test\injury-prediction\.venv\Lib\site-packages\open_clip\factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


[INFO] Loading CalorieCLIP into memory on device: cpu...
[INFO] Model loaded successfully!
Estimated: 218 calories


Estimating calories for each player's photos, extracting photo dates, creating individual DataFrames, and concatenating them into a single DataFrame:

In [4]:
from pathlib import Path
from PIL import Image
import pandas as pd
from utils.calorie_estimation import CalorieEstimation

# Initialize CalorieEstimation using the utility class
model = CalorieEstimation()

players = ["p01", "p03", "p05"]
player_dfs = []
BATCH_SIZE = 32

for player in players:
    player_path = Path(f"data/{player}/food-images")
    if not player_path.exists():
        continue
    
    image_paths = []
    for ext in ("*.jpg", "*.jpeg", "*.JPG", "*.JPEG"):
        image_paths.extend(list(player_path.glob(ext)))
    
    if not image_paths:
        continue
        
    dates = []
    calories_list = []
    
    for i in range(0, len(image_paths), BATCH_SIZE):
        batch_paths = image_paths[i:i + BATCH_SIZE]
        calories_batch = model.predict_batch(batch_paths)
        
        for img_path, cals in zip(batch_paths, calories_batch):
            img = Image.open(img_path)
            exif = img.getexif()
            dt = None
            if exif:
                dt_str = exif.get(306)
                if 34665 in exif:
                    sub_exif = exif.get_ifd(34665)
                    dt_str = sub_exif.get(36867, dt_str)
                if dt_str:
                    try:
                        dt = dt_str.split(' ')[0].replace(':', '-')
                    except Exception:
                        dt = dt_str
            if not dt:
                dt = pd.to_datetime(img_path.stat().st_mtime, unit='s').strftime('%Y-%m-%d')
                
            dates.append(dt)
            calories_list.append(float(cals))
            
    df_player = pd.DataFrame({
        'player': player,
        'image': [p.name for p in image_paths],
        'date': dates,
        'calories': calories_list
    })
    player_dfs.append(df_player)

# Concatenate all 3 player DataFrames into 1
food_calories_df = pd.concat(player_dfs, ignore_index=True)

[INFO] Downloading/Verifying CalorieCLIP model files locally...


Fetching 19 files: 100%|██████████| 19/19 [00:00<00:00, 73.53it/s]


[INFO] Loading CalorieCLIP into memory on device: cpu...


c:\Users\paris\Documents\Cours Centrale 3A\technical-test\injury-prediction\.venv\Lib\site-packages\open_clip\factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


[INFO] Model loaded successfully!


In [5]:
print(food_calories_df)

     player          image        date    calories
0       p01  IMG_8916.jpeg  2020-02-01  218.451416
1       p01  IMG_8917.jpeg  2020-02-01  447.812347
2       p01  IMG_8918.jpeg  2020-02-01  317.138062
3       p01  IMG_8920.jpeg  2020-02-01  430.066254
4       p01  IMG_8921.jpeg  2020-02-01  203.444107
...     ...            ...         ...         ...
1279    p05   IMG_2924.jpg  2020-03-26  297.093719
1280    p05   IMG_2925.jpg  2020-03-26  240.066147
1281    p05   IMG_2931.jpg  2020-03-27  271.975830
1282    p05   IMG_2966.jpg  2020-03-28  315.789917
1283    p05   IMG_2970.jpg  2020-03-29  578.645020

[1284 rows x 4 columns]


## Database aggregation

In [8]:
import ast
from pathlib import Path
import pandas as pd

# Aggregate food_calories_df per player and date to get total calories eaten
food_calories_agg = food_calories_df.groupby(['player', 'date'])['calories'].sum().reset_index().rename(columns={'calories': 'total_calories_eaten'})

root_path = Path("data").resolve()
players = [p.name for p in root_path.iterdir() if p.is_dir() and (p / "pmsys" / "wellness.csv").exists()]

all_player_dfs = []

for player in players:
    p = root_path / player
    dates = set()
    
    def add_dates(series):
        dt = pd.to_datetime(series, errors="coerce").dt.strftime("%Y-%m-%d")
        dates.update(dt.dropna().tolist())
        
    if (p / "pmsys" / "wellness.csv").exists():
        add_dates(pd.read_csv(p / "pmsys" / "wellness.csv")["effective_time_frame"])
    if (p / "googledocs" / "reporting.csv").exists():
        add_dates(pd.to_datetime(pd.read_csv(p / "googledocs" / "reporting.csv")["date"], format="%d/%m/%Y", errors="coerce"))
    if (p / "fitbit" / "calories.json").exists():
        add_dates(pd.read_json(p / "fitbit" / "calories.json")["dateTime"])
    if (p / "fitbit" / "sleep.json").exists():
        df_s = pd.read_json(p / "fitbit" / "sleep.json")
        if "dateOfSleep" in df_s.columns:
            dates.update(df_s["dateOfSleep"].dropna().tolist())
    if (p / "pmsys" / "srpe.csv").exists():
        add_dates(pd.read_csv(p / "pmsys" / "srpe.csv")["end_date_time"])
    if (p / "pmsys" / "injury.csv").exists():
        add_dates(pd.read_csv(p / "pmsys" / "injury.csv")["effective_time_frame"])
        
    if not dates:
        continue
        
    df_player = pd.DataFrame({'player': player, 'date': sorted(list(dates))})
    
    # 1. calories.json -> "total calories consumed"
    if (p / "fitbit" / "calories.json").exists():
        df_cals = pd.read_json(p / "fitbit" / "calories.json")
        df_cals['date'] = pd.to_datetime(df_cals['dateTime']).dt.strftime('%Y-%m-%d')
        df_cals['value'] = df_cals['value'].astype(float)
        df_cals_agg = df_cals.groupby('date')['value'].sum().reset_index().rename(columns={'value': 'total calories consumed'})
        df_player = pd.merge(df_player, df_cals_agg, on='date', how='left')
        
    # 2. exercise.json -> activity zones
    if (p / "fitbit" / "exercise.json").exists():
        ex = pd.read_json(p / "fitbit" / "exercise.json")
        ex_rows = []
        for idx, row in ex.iterrows():
            dt = pd.to_datetime(row['startTime']).strftime('%Y-%m-%d')
            levels = {lvl['name']: lvl['minutes'] for lvl in row.get('activityLevel', [])}
            ex_rows.append({
                'date': dt,
                'exercise_time_in_zone_sedentary': levels.get('sedentary', 0),
                'exercise_time_in_zone_lightly': levels.get('lightly', 0),
                'exercise_time_in_zone_fairly': levels.get('fairly', 0),
                'exercise_time_in_zone_very': levels.get('very', 0),
            })
        if ex_rows:
            df_ex = pd.DataFrame(ex_rows).groupby('date').sum().reset_index()
            df_player = pd.merge(df_player, df_ex, on='date', how='left')
            
    # 3. heart_rate.json -> max_heart_rate
    if (p / "fitbit" / "heart_rate.json").exists():
        df_hr = pd.read_json(p / "fitbit" / "heart_rate.json")
        df_hr['date'] = pd.to_datetime(df_hr['dateTime']).dt.strftime('%Y-%m-%d')
        df_hr['bpm'] = df_hr['value'].apply(lambda x: x.get('bpm') if isinstance(x, dict) else None)
        df_hr_agg = df_hr.groupby('date')['bpm'].max().reset_index().rename(columns={'bpm': 'max_heart_rate'})
        df_player = pd.merge(df_player, df_hr_agg, on='date', how='left')
        
    # 4. resting_heart_rate.json -> resting_heart_rate
    if (p / "fitbit" / "resting_heart_rate.json").exists():
        df_rhr = pd.read_json(p / "fitbit" / "resting_heart_rate.json")
        df_rhr['date'] = pd.to_datetime(df_rhr['dateTime']).dt.strftime('%Y-%m-%d')
        df_rhr['resting_heart_rate'] = df_rhr['value'].apply(lambda x: x.get('value') if isinstance(x, dict) else None)
        df_rhr_agg = df_rhr[['date', 'resting_heart_rate']].groupby('date').min().reset_index()
        df_player = pd.merge(df_player, df_rhr_agg, on='date', how='left')
        
    # 5. sleep_score.csv -> sleep_score
    if (p / "fitbit" / "sleep_score.csv").exists():
        df_ss = pd.read_csv(p / "fitbit" / "sleep_score.csv")
        df_ss['date'] = pd.to_datetime(df_ss['timestamp']).dt.strftime('%Y-%m-%d')
        df_ss = df_ss[['date', 'overall_score']].groupby('date').mean().reset_index().rename(columns={'overall_score': 'sleep_score'})
        df_player = pd.merge(df_player, df_ss, on='date', how='left')
        
    # 6. sleep.json -> sleep_duration, sleep_start_time, sleep_end_time
    if (p / "fitbit" / "sleep.json").exists():
        df_sleep = pd.read_json(p / "fitbit" / "sleep.json")
        sleep_rows = []
        for idx, row in df_sleep.iterrows():
            dt = row.get('dateOfSleep')
            duration_ms = row.get('duration', 0)
            duration_h = duration_ms / 3600000.0
            start_time = row.get('startTime')
            end_time = row.get('endTime')
            sleep_rows.append({
                'date': dt,
                'sleep_duration': duration_h,
                'sleep_start_time': start_time,
                'sleep_end_time': end_time
            })
        if sleep_rows:
            df_sleep_agg = pd.DataFrame(sleep_rows).groupby('date').agg({
                'sleep_duration': 'sum',
                'sleep_start_time': 'first',
                'sleep_end_time': 'last'
            }).reset_index()
            df_player = pd.merge(df_player, df_sleep_agg, on='date', how='left')
            
    # 7. steps.json -> total_steps
    if (p / "fitbit" / "steps.json").exists():
        df_steps = pd.read_json(p / "fitbit" / "steps.json")
        df_steps['date'] = pd.to_datetime(df_steps['dateTime']).dt.strftime('%Y-%m-%d')
        df_steps['value'] = df_steps['value'].astype(float)
        df_steps_agg = df_steps.groupby('date')['value'].sum().reset_index().rename(columns={'value': 'total_steps'})
        df_player = pd.merge(df_player, df_steps_agg, on='date', how='left')
        
    # 8. time_in_heart_rate_zones.json -> heart rate zones
    if (p / "fitbit" / "time_in_heart_rate_zones.json").exists():
        df_hrz = pd.read_json(p / "fitbit" / "time_in_heart_rate_zones.json")
        df_hrz['date'] = pd.to_datetime(df_hrz['dateTime']).dt.strftime('%Y-%m-%d')
        zones_list = [item.get('valuesInZones', {}) for item in df_hrz['value']]
        df_zones = pd.DataFrame(zones_list)
        df_zones['date'] = df_hrz['date']
        df_zones_agg = df_zones.groupby('date').sum().reset_index()
        df_player = pd.merge(df_player, df_zones_agg, on='date', how='left')
        
    # 9. reporting.csv -> weight, meals, glasses_of_fluid, alcohol_consumed
    if (p / "googledocs" / "reporting.csv").exists():
        df_rep = pd.read_csv(p / "googledocs" / "reporting.csv")
        df_rep['date'] = pd.to_datetime(df_rep['date'], format='%d/%m/%Y', errors='coerce').dt.strftime('%Y-%m-%d')
        for meal in ['Breakfast', 'Lunch', 'Dinner', 'Evening']:
            df_rep[meal] = df_rep['meals'].apply(lambda x: 1 if isinstance(x, str) and meal in x else 0)
        df_rep = df_rep.drop(columns=['timestamp', 'meals'], errors='ignore')
        df_rep_agg = df_rep.groupby('date').last().reset_index()
        df_player = pd.merge(df_player, df_rep_agg, on='date', how='left')
        
    # 10. injury.csv -> minor_injuries, major_injuries
    if (p / "pmsys" / "injury.csv").exists():
        df_inj = pd.read_csv(p / "pmsys" / "injury.csv")
        df_inj['date'] = pd.to_datetime(df_inj['effective_time_frame']).dt.strftime('%Y-%m-%d')
        def parse_injury(x):
            minor = []
            major = []
            try:
                d = ast.literal_eval(x) if isinstance(x, str) else x
                if isinstance(d, dict):
                    for loc, grav in d.items():
                        if str(grav).lower() == 'minor':
                            minor.append(loc)
                        else:
                            major.append(loc)
            except:
                pass
            return pd.Series([minor, major])
        df_inj[['minor_injuries', 'major_injuries']] = df_inj['injuries'].apply(parse_injury)
        df_inj_agg = df_inj.groupby('date').agg({
            'minor_injuries': lambda x: list(dict.fromkeys([loc for lst in x for loc in lst])),
            'major_injuries': lambda x: list(dict.fromkeys([loc for lst in x for loc in lst]))
        }).reset_index()
        df_player = pd.merge(df_player, df_inj_agg, on='date', how='left')
                
    # 11. srpe.csv -> effort_end_datetime, sRPE
    if (p / "pmsys" / "srpe.csv").exists():
        df_srpe = pd.read_csv(p / "pmsys" / "srpe.csv")
        df_srpe['date'] = pd.to_datetime(df_srpe['end_date_time']).dt.strftime('%Y-%m-%d')
        df_srpe['sRPE'] = df_srpe['perceived_exertion'] * df_srpe['duration_min']
        df_srpe_agg = df_srpe.groupby('date').agg({
            'end_date_time': 'last',
            'sRPE': 'sum'
        }).reset_index().rename(columns={'end_date_time': 'effort_end_datetime'})
        df_player = pd.merge(df_player, df_srpe_agg, on='date', how='left')
        
    # 12. wellness.csv -> all data except effective_timeframe and soreness_area
    if (p / "pmsys" / "wellness.csv").exists():
        df_wel = pd.read_csv(p / "pmsys" / "wellness.csv")
        df_wel['date'] = pd.to_datetime(df_wel['effective_time_frame']).dt.strftime('%Y-%m-%d')
        df_wel = df_wel.drop(columns=['effective_time_frame', 'soreness_area'], errors='ignore')
        df_wel_agg = df_wel.groupby('date').last().reset_index()
        df_player = pd.merge(df_player, df_wel_agg, on='date', how='left')
        
    all_player_dfs.append(df_player)

master_df = pd.concat(all_player_dfs, ignore_index=True)

# Merge food calories eaten from food_calories_agg
master_df = pd.merge(master_df, food_calories_agg, on=['player', 'date'], how='left')

In [9]:
print(master_df.head())
print("Master DataFrame shape:", master_df.shape)

  player        date  total calories consumed  \
0    p01  2019-11-01                  4009.10   
1    p01  2019-11-02                  3533.56   
2    p01  2019-11-03                  3748.73   
3    p01  2019-11-04                  3353.38   
4    p01  2019-11-05                  3794.63   

   exercise_time_in_zone_sedentary  exercise_time_in_zone_lightly  \
0                              0.0                            1.0   
1                              0.0                            0.0   
2                              NaN                            NaN   
3                              0.0                            1.0   
4                              3.0                           13.0   

   exercise_time_in_zone_fairly  exercise_time_in_zone_very  max_heart_rate  \
0                          20.0                        37.0           140.0   
1                          11.0                        30.0           122.0   
2                           NaN                      